In [ ]:
# Core
import numpy as np
import pandas as pd
import os
import pyarrow.parquet as pq
from pathlib import Path
# Models
import lightgbm as lgb
import xgboost as xgb

# Sklearn utilities
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Visualization (for EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Kaggle-specific
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

#visualize
import pandas as pd
import textwrap
import plotly.graph_objects as go
from pathlib import Path

ModuleNotFoundError: No module named 'Path'

In [2]:
bars_seen_private_test = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/bars_seen_private_test.parquet")
bars_seen_public_test = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/bars_seen_public_test.parquet")
bars_seen_train = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/bars_seen_train.parquet")
bars_unseen_train = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/bars_unseen_train.parquet")
headlines_seen_private_test = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_private_test.parquet")
headlines_seen_public_test = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_public_test.parquet")
headlines_seen_train = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_train.parquet")
headlines_unseen_train = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_unseen_train.parquet")


In [3]:
import pandas as pd
import textwrap
import plotly.graph_objects as go
from pathlib import Path

# Notebook-safe path resolution
if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parent
else:
    cwd = Path.cwd().resolve()
    # If notebook is launched from workspace root, move into datathon2026
    if (cwd / "datathon2026").exists() and (cwd / "hrt-eth-zurich-datathon-2026").exists():
        BASE_DIR = cwd / "datathon2026"
    else:
        BASE_DIR = cwd

candidate_data_dirs = [
    BASE_DIR / "../hrt-eth-zurich-datathon-2026/data",
    BASE_DIR.parent / "hrt-eth-zurich-datathon-2026/data",
]

DATA_DIR = next((p.resolve() for p in candidate_data_dirs if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data directory. Tried:\n" + "\n".join(str(p.resolve()) for p in candidate_data_dirs)
    )

OUT_DIR = (BASE_DIR / "visualization").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

seen = pd.read_parquet(DATA_DIR / "bars_seen_train.parquet")
unseen = pd.read_parquet(DATA_DIR / "bars_unseen_train.parquet")
hl_seen = pd.read_parquet(DATA_DIR / "headlines_seen_train.parquet")
hl_unseen = pd.read_parquet(DATA_DIR / "headlines_unseen_train.parquet")

full = pd.concat([seen.assign(part="seen"), unseen.assign(part="unseen")], ignore_index=True)
full = full.sort_values(["session", "bar_ix"]).reset_index(drop=True)
headlines = pd.concat([hl_seen.assign(part="seen"), hl_unseen.assign(part="unseen")],
                      ignore_index=True).sort_values(["session", "bar_ix"]).reset_index(drop=True)

split_ix = int(seen["bar_ix"].max())  # last seen bar index (49)
N_SESSIONS = 30
SESSIONS = sorted(full["session"].unique().tolist())[:N_SESSIONS]


def session_traces(session_id):
    s = full[full["session"] == session_id]
    s_seen = s[s["part"] == "seen"]
    s_unseen = s[s["part"] == "unseen"]

    traces = []

    traces.append(go.Candlestick(
        x=s_seen["bar_ix"], open=s_seen["open"], high=s_seen["high"],
        low=s_seen["low"], close=s_seen["close"],
        name="seen",
        increasing_line_color="#1f77b4", decreasing_line_color="#1f77b4",
        increasing_fillcolor="#1f77b4", decreasing_fillcolor="#9ecae1",
        showlegend=True,
    ))
    traces.append(go.Candlestick(
        x=s_unseen["bar_ix"], open=s_unseen["open"], high=s_unseen["high"],
        low=s_unseen["low"], close=s_unseen["close"],
        name="unseen",
        increasing_line_color="#d62728", decreasing_line_color="#d62728",
        increasing_fillcolor="#d62728", decreasing_fillcolor="#fcae91",
        showlegend=True,
    ))

    hs = headlines[headlines["session"] == session_id]
    if len(hs):
        hi = float(s["high"].max())
        lo = float(s["low"].min())
        span = hi - lo if hi > lo else 1.0
        top_y = hi + span * 0.10

        bar_high = s.set_index("bar_ix")["high"]

        for part_label, color in [("seen", "#1f77b4"), ("unseen", "#d62728")]:
            sub = hs[hs["part"] == part_label]
            if not len(sub):
                continue

            line_x, line_y = [], []
            for row in sub.itertuples(index=False):
                hb = float(bar_high.get(row.bar_ix, hi))
                line_x += [row.bar_ix, row.bar_ix, None]
                line_y += [top_y, hb, None]
            traces.append(go.Scatter(
                x=line_x, y=line_y, mode="lines",
                line=dict(color=color, width=1.2),
                hoverinfo="skip", showlegend=False,
            ))

            marker_x = sub["bar_ix"].tolist()
            marker_y = [top_y] * len(sub)
            wrapped = ["<br>".join(textwrap.wrap(h, 60)) for h in sub["headline"]]
            cd = [(int(b), part_label) for b in sub["bar_ix"]]
            traces.append(go.Scatter(
                x=marker_x, y=marker_y, mode="markers",
                marker=dict(symbol="triangle-down", size=14, color=color,
                            line=dict(width=1, color="black")),
                text=wrapped,
                customdata=cd,
                hovertemplate="<b>bar %{customdata[0]} (%{customdata[1]})</b><br>%{text}<extra></extra>",
                name=f"headlines ({part_label})",
                hoverlabel=dict(bgcolor="white", font_size=11),
            ))
    return traces


fig = go.Figure()
trace_session = []
for sid in SESSIONS:
    ts = session_traces(sid)
    for t in ts:
        fig.add_trace(t)
        trace_session.append(sid)

default_sid = SESSIONS[0]
for i, sid in enumerate(trace_session):
    fig.data[i].visible = (sid == default_sid)

buttons = []
for sid in SESSIONS:
    visibility = [s == sid for s in trace_session]
    buttons.append(dict(
        label=f"Session {sid}",
        method="update",
        args=[{"visible": visibility},
              {"title.text": f"Session {sid} - OHLC + headlines (hover markers)"}],
    ))

fig.update_layout(
    title=f"Session {default_sid} - OHLC + headlines (hover markers)",
    xaxis_title="bar_ix",
    yaxis_title="price",
    xaxis_rangeslider_visible=False,
    height=650,
    updatemenus=[dict(
        active=0, buttons=buttons, x=1.02, xanchor="left", y=1, yanchor="top",
        showactive=True,
    )],
    shapes=[dict(
        type="line", xref="x", yref="paper",
        x0=split_ix + 0.5, x1=split_ix + 0.5, y0=0, y1=1,
        line=dict(color="black", dash="dash", width=1),
    )],
    annotations=[dict(
        x=split_ix + 0.5, y=1.02, xref="x", yref="paper",
        text="seen | unseen", showarrow=False, font=dict(size=10),
    )],
)

out_html = OUT_DIR / "candles_interactive.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Wrote interactive plot: {out_html}")
import sys
if "ipykernel" in sys.modules:
    fig.show()
else:
    print("Run this cell in a notebook to view the interactive chart inline.")




Wrote interactive plot: /Users/lukawedegartner/Documents/eth/Datathon/datathon2026/visualization/candles_interactive.html


In [ ]:
# OpenAI title extraction skeleton
import os
import json
import asyncio
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI

load_dotenv()  # loads .env if present

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set in your environment.")

client = OpenAI()
aclient = AsyncOpenAI()
MODEL = "gpt-5.3-codex"  # GPT-5.3 family; change if you prefer another 5.3 variant
OUTPUT_PATH = Path("title_extractions_gpt53.json")


In [1]:
# OpenAI schema + template-index extraction
TEMPLATE_FILE = Path("headline_templates.txt")
if not TEMPLATE_FILE.exists():
    raise FileNotFoundError(
        f"{TEMPLATE_FILE.resolve()} not found. Run the template extraction cell first."
    )

TEMPLATE_CATALOG = []
with TEMPLATE_FILE.open("r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        line = line.rstrip("\n")
        if not line:
            continue
        count_str, template = line.split("\t", 1)
        TEMPLATE_CATALOG.append({
            "index": idx,
            "count": int(count_str),
            "template": template,
        })

if not TEMPLATE_CATALOG:
    raise ValueError("No templates found in headline_templates.txt")

TEMPLATE_LOOKUP = {item["index"]: item["template"] for item in TEMPLATE_CATALOG}
TEMPLATE_INDEX_TEXT = "\n".join(
    f"{item['index']}: {item['template']}" for item in TEMPLATE_CATALOG
)

SCHEMA_TEMPLATE = {
    "title": "",
    "template_index": -1,
    "percentage": "",
    "dollar": "",
    "company": "",
    "sector": "",
    "region": ""
}

SYSTEM_PROMPT = (
    "You extract structured data from financial headlines. "
    "Return ONLY valid JSON with exactly these keys: "
    + json.dumps(SCHEMA_TEMPLATE)
)

def _strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return text

def _coerce_record(parsed: dict, title: str) -> dict:
    out = {
        "title": title,
        "template_index": -1,
        "percentage": "",
        "dollar": "",
        "company": "",
        "sector": "",
        "region": "",
    }

    out.update({k: parsed.get(k, out[k]) for k in out.keys() if k in parsed})

    try:
        out["template_index"] = int(out["template_index"])
    except Exception:
        out["template_index"] = -1

    if out["template_index"] not in TEMPLATE_LOOKUP:
        out["template_index"] = -1

    for key in ["percentage", "dollar", "company", "sector", "region"]:
        value = out.get(key, "")
        out[key] = "" if value is None else str(value).strip()

    out["template_text"] = TEMPLATE_LOOKUP.get(out["template_index"], "")
    return out

def _build_user_prompt(title: str) -> str:
    return (
        f"Headline: {title}\n\n"
        "Template catalog (index: template):\n"
        f"{TEMPLATE_INDEX_TEXT}\n\n"
        "Instructions:\n"
        "1) Choose template_index from the catalog for the best matching template.\n"
        "2) Extract percentage, dollar, company, sector, region from the headline.\n"
        "3) If a field is absent, return empty string for that field."
    )

async def extract_title_info_async(title: str, retries: int = 3) -> dict:
    user_prompt = _build_user_prompt(title)
    raw_text = ""

    for attempt in range(1, retries + 1):
        try:
            response = await aclient.responses.create(
                model=MODEL,
                instructions=SYSTEM_PROMPT,
                input=user_prompt,
                temperature=0,
            )

            raw_text = response.output_text or ""
            cleaned = _strip_code_fences(raw_text)
            parsed = json.loads(cleaned)
            if not isinstance(parsed, dict):
                raise ValueError("Model output is not a JSON object")
            return _coerce_record(parsed, title)
        except Exception as exc:
            if attempt < retries:
                await asyncio.sleep(min(2 ** (attempt - 1), 8))
                continue
            return {
                "title": title,
                "template_index": -1,
                "template_text": "",
                "percentage": "",
                "dollar": "",
                "company": "",
                "sector": "",
                "region": "",
                "api_error": str(exc),
                "raw_response": raw_text,
            }


NameError: name 'Path' is not defined

In [ ]:
# Run extraction for every title and save JSON (parallel with semaphore)
from tqdm.auto import tqdm

SOURCE_DF = pd.concat([
    pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_train.parquet"),
    pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_unseen_train.parquet"),
], ignore_index=True)

TITLE_COLUMN = "headline"
MAX_TITLES = None  # set to an int for quick tests, e.g. 100
MAX_CONCURRENCY = 50
REQUEST_RETRIES = 3
SLEEP_SECONDS = 0.0  # optional per-task delay after each request

OUTPUT_PATH = Path("title_template_slot_extractions_gpt53.json")

work_df = SOURCE_DF.dropna(subset=[TITLE_COLUMN]).copy().reset_index(drop=True)
if MAX_TITLES is not None:
    work_df = work_df.head(MAX_TITLES)

async def _extract_one(i: int, row: pd.Series, semaphore: asyncio.Semaphore):
    title = str(row[TITLE_COLUMN])

    async with semaphore:
        record = await extract_title_info_async(title, retries=REQUEST_RETRIES)

    record["row_index"] = int(i)
    if "session" in row.index and pd.notna(row["session"]):
        record["session"] = int(row["session"])
    if "bar_ix" in row.index and pd.notna(row["bar_ix"]):
        record["bar_ix"] = int(row["bar_ix"])

    if SLEEP_SECONDS > 0:
        await asyncio.sleep(SLEEP_SECONDS)

    return i, record

async def _run_parallel(df: pd.DataFrame):
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = [
        asyncio.create_task(_extract_one(i, row, semaphore))
        for i, row in df.iterrows()
    ]

    ordered = [None] * len(df)
    with tqdm(total=len(df), desc=f"OpenAI extraction ({MAX_CONCURRENCY} concurrent)", unit="title") as pbar:
        for fut in asyncio.as_completed(tasks):
            i, record = await fut
            ordered[i] = record
            pbar.update(1)

    return ordered

def _run_coro(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    import nest_asyncio
    nest_asyncio.apply(loop)
    return loop.run_until_complete(coro)

results = _run_coro(_run_parallel(work_df))

OUTPUT_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"Saved {len(results)} records to {OUTPUT_PATH.resolve()}")

pd.DataFrame(results).head()


In [ ]:
# Headline template extraction
import re
from pathlib import Path

SECTOR_PHRASES = [
    "wireless connectivity", "cloud infrastructure", "enterprise software",
    "renewable storage", "renewable energy", "automated logistics",
    "precision medicine", "supply chain optimization", "supply chain",
    "data analytics", "advanced manufacturing", "consumer electronics",
    "biotechnology", "fintech", "robotics", "autonomous vehicles",
    "precision manufacturing",
    "precision manufacturing systems",
    "process automation",
    "process automation systems",
    "digital payments",
    "digital payments systems",
]
REGIONS = [
    "Southeast Asia", "Asia Pacific", "Latin America", "Central Europe",
    "North America", "Eastern Europe", "Middle East", "Scandinavia",
    "Africa", "South America", "Europe",
]

PARTNER_PHRASES = [
    "a multinational manufacturer",
    "an international consortium",
    "a top-tier research institute",
    "a leading cloud platform",
    "a major logistics provider",
    "a global retailer",
    "a national infrastructure agency",
]

def extract_template(text: str) -> str:
    t = text
    t = re.sub(r"\$\d+(?:\.\d+)?\s?[BMK]?", "$X", t)
    t = re.sub(r"\d+(?:\.\d+)?\s?%", "N%", t)
    t = re.sub(r"\b\d+(?:\.\d+)?\b", "N", t)
    t = re.sub(r"^(?:[A-Z][A-Za-z]+\s){1,3}", "<COMPANY> ", t)
    t = re.sub(r"^(<COMPANY>\s+).+?\s+steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)(?:.+?\s+)?addresses investor concerns in open letter$", r"\1<ROLE> addresses investor concerns in open letter", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)to present at\s+.+$", r"\1to present at <EVENT>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)confirms participation in\s+.+$", r"\1confirms participation in <EVENT>", t, flags=re.IGNORECASE)
    for p in PARTNER_PHRASES:
        t = re.sub(re.escape(p), "<PARTNER>", t, flags=re.IGNORECASE)
    for s in SECTOR_PHRASES:
        t = re.sub(re.escape(s), "<SECTOR>", t, flags=re.IGNORECASE)
    for r in REGIONS:
        t = re.sub(re.escape(r), "<REGION>", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip()
    return t

hl = pd.concat([
    pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_train.parquet"),
    pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_unseen_train.parquet"),
], ignore_index=True)

hl["template"] = hl["headline"].astype(str).map(extract_template)
template_counts = hl["template"].value_counts()

out_path = Path("headline_templates.txt")
with out_path.open("w", encoding="utf-8") as f:
    for template, count in template_counts.items():
        f.write(f"{count}\t{template}\n")

print(f"Headlines: {len(hl)}")
print(f"Distinct templates: {hl['template'].nunique()}")
print(f"Saved templates to: {out_path.resolve()}")
template_counts.head(20)




In [24]:
# Rule-based parser for template/slot JSON (no API calls)
import json
import pandas as pd
import re
from pathlib import Path

# Choose source headlines
DATASET = "public"  # "train", "public", "private"

if DATASET == "train":
    SOURCE_DF = pd.concat([
        pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_train.parquet"),
        pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_unseen_train.parquet"),
    ], ignore_index=True)
elif DATASET == "public":
    SOURCE_DF = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_public_test.parquet")
elif DATASET == "private":
    SOURCE_DF = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_private_test.parquet")
else:
    raise ValueError("DATASET must be one of: train, public, private")

# Safety: if a trailing comma accidentally makes SOURCE_DF a tuple, unwrap it.
if isinstance(SOURCE_DF, tuple):
    if len(SOURCE_DF) == 1 and isinstance(SOURCE_DF[0], pd.DataFrame):
        SOURCE_DF = SOURCE_DF[0]
    else:
        raise TypeError("SOURCE_DF is a tuple; remove trailing comma from assignment")

TEMPLATE_FILE = Path("headline_templates.txt")
if not TEMPLATE_FILE.exists():
    raise FileNotFoundError(
        f"{TEMPLATE_FILE.resolve()} not found. Run the template extraction cell first."
    )

SECTOR_PHRASES = [
    "wireless connectivity", "cloud infrastructure", "enterprise software",
    "renewable storage", "renewable energy", "automated logistics",
    "precision medicine", "supply chain optimization", "supply chain",
    "data analytics", "advanced manufacturing", "consumer electronics",
    "biotechnology", "fintech", "robotics", "autonomous vehicles",
    "precision manufacturing", "process automation", "digital payments",
    "precision manufacturing systems", "process automation systems", "digital payments systems",
]
REGIONS = [
    "Southeast Asia", "Asia Pacific", "Latin America", "Central Europe",
    "North America", "Eastern Europe", "Middle East", "Scandinavia",
    "Africa", "South America", "Europe",
]

PARTNER_PHRASES = [
    "a multinational manufacturer",
    "an international consortium",
    "a top-tier research institute",
    "a leading cloud platform",
    "a major logistics provider",
    "a global retailer",
    "a national infrastructure agency",
]

DOLLAR_RE = re.compile(r"\$\d+(?:\.\d+)?\s?[BMK]?")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?\s?%")
COMPANY_RE = re.compile(r"^([A-Z][A-Za-z]*(?:\s+[A-Z][A-Za-z]*){0,2})\b")

SECTOR_LOOKUP = sorted(SECTOR_PHRASES, key=len, reverse=True)
REGION_LOOKUP = sorted(REGIONS, key=len, reverse=True)

def _find_phrase(text: str, phrases: list[str]) -> str:
    low = text.lower()
    for phrase in phrases:
        if phrase.lower() in low:
            return phrase
    return ""

def extract_template(text: str) -> str:
    t = text
    t = re.sub(r"\$\d+(?:\.\d+)?\s?[BMK]?", "$X", t)
    t = re.sub(r"\d+(?:\.\d+)?\s?%", "N%", t)
    t = re.sub(r"\b\d+(?:\.\d+)?\b", "N", t)
    t = re.sub(r"^(?:[A-Z][A-Za-z]+\s){1,3}", "<COMPANY> ", t)
    t = re.sub(r"^(<COMPANY>\s+).+?\s+steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)(?:.+?\s+)?addresses investor concerns in open letter$", r"\1<ROLE> addresses investor concerns in open letter", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)to present at\s+.+$", r"\1to present at <EVENT>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)confirms participation in\s+.+$", r"\1confirms participation in <EVENT>", t, flags=re.IGNORECASE)
    for p in PARTNER_PHRASES:
        t = re.sub(re.escape(p), "<PARTNER>", t, flags=re.IGNORECASE)
    for s in SECTOR_PHRASES:
        t = re.sub(re.escape(s), "<SECTOR>", t, flags=re.IGNORECASE)
    for r in REGIONS:
        t = re.sub(re.escape(r), "<REGION>", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip()
    return t

template_to_index = {}
index_to_template = {}
with TEMPLATE_FILE.open("r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        line = line.rstrip("\n")
        if not line:
            continue
        _, template = line.split("\t", 1)
        template_to_index[template] = idx
        index_to_template[idx] = template

def parse_title(title: str) -> dict:
    title = str(title)
    template = extract_template(title)
    template_index = template_to_index.get(template, -1)

    m_pct = PERCENT_RE.search(title)
    m_dol = DOLLAR_RE.search(title)
    m_company = COMPANY_RE.match(title.strip())

    record = {
        "title": title,
        "template_index": int(template_index),
        "template_text": index_to_template.get(template_index, ""),
        "percentage": m_pct.group(0).strip() if m_pct else "",
        "dollar": m_dol.group(0).strip() if m_dol else "",
        "company": m_company.group(1).strip() if m_company else "",
        "sector": _find_phrase(title, SECTOR_LOOKUP),
        "region": _find_phrase(title, REGION_LOOKUP),
    }
    return record

MAX_TITLES = None  # set to int for testing
OUTPUT_NAME_MAP = {
    "train": "title_template_slot_extractions_parser.json",
    "public": "title_template_slot_extractions_parser_public.json",
    "private": "title_template_slot_extractions_parser_private.json",
}
OUTPUT_PATH = Path(OUTPUT_NAME_MAP[DATASET])

work_df = SOURCE_DF.dropna(subset=["headline"]).copy().reset_index(drop=True)
if MAX_TITLES is not None:
    work_df = work_df.head(MAX_TITLES)

results = []
for i, row in work_df.iterrows():
    rec = parse_title(row["headline"])
    rec["row_index"] = int(i)
    if "session" in row.index and pd.notna(row["session"]):
        rec["session"] = int(row["session"])
    if "bar_ix" in row.index and pd.notna(row["bar_ix"]):
        rec["bar_ix"] = int(row["bar_ix"])
    results.append(rec)

OUTPUT_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"Dataset={DATASET} | Saved {len(results)} records to {OUTPUT_PATH.resolve()}")
pd.DataFrame(results).head()


Saved 17371 records to /Users/lukawedegartner/Documents/eth/Datathon/title_template_slot_extractions_parser.json


,title,template_index,template_text,percentage,dollar,company,sector,region,row_index,session,bar_ix
0,Relvos Biosciences opens new office in Southea...,20,<COMPANY> opens new office in <REGION>,,,Relvos Biosciences,,Southeast Asia,0,0,6
1,Orevex Renewables secures $500M contract with ...,0,<COMPANY> secures $X contract with <PARTNER>,,$500M,Orevex Renewables,,,1,0,12
2,Relvos Biosciences names new head of precision...,23,<COMPANY> names new head of <SECTOR> division,,,Relvos Biosciences,precision manufacturing,,2,0,14
3,Calvis Sciences secures $650M contract with a ...,0,<COMPANY> secures $X contract with <PARTNER>,,$650M,Calvis Sciences,,,3,0,20
4,Yorvov Pharmaceuticals secures $180M contract ...,0,<COMPANY> secures $X contract with <PARTNER>,,$180M,Yorvov Pharmaceuticals,,,4,0,22


In [25]:
# Rule-based parser for template/slot JSON (no API calls)
import json
import pandas as pd
import re
from pathlib import Path

# Choose source headlines
DATASET = "public"  # "train", "public", "private"

if DATASET == "train":
    SOURCE_DF = pd.concat([
        pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_train.parquet"),
        pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_unseen_train.parquet"),
    ], ignore_index=True)
elif DATASET == "public":
    SOURCE_DF = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_public_test.parquet")
elif DATASET == "private":
    SOURCE_DF = pd.read_parquet("hrt-eth-zurich-datathon-2026/data/headlines_seen_private_test.parquet")
else:
    raise ValueError("DATASET must be one of: train, public, private")

# Safety: if a trailing comma accidentally makes SOURCE_DF a tuple, unwrap it.
if isinstance(SOURCE_DF, tuple):
    if len(SOURCE_DF) == 1 and isinstance(SOURCE_DF[0], pd.DataFrame):
        SOURCE_DF = SOURCE_DF[0]
    else:
        raise TypeError("SOURCE_DF is a tuple; remove trailing comma from assignment")

TEMPLATE_FILE = Path("headline_templates.txt")
if not TEMPLATE_FILE.exists():
    raise FileNotFoundError(
        f"{TEMPLATE_FILE.resolve()} not found. Run the template extraction cell first."
    )

SECTOR_PHRASES = [
    "wireless connectivity", "cloud infrastructure", "enterprise software",
    "renewable storage", "renewable energy", "automated logistics",
    "precision medicine", "supply chain optimization", "supply chain",
    "data analytics", "advanced manufacturing", "consumer electronics",
    "biotechnology", "fintech", "robotics", "autonomous vehicles",
    "precision manufacturing", "process automation", "digital payments",
    "precision manufacturing systems", "process automation systems", "digital payments systems",
]
REGIONS = [
    "Southeast Asia", "Asia Pacific", "Latin America", "Central Europe",
    "North America", "Eastern Europe", "Middle East", "Scandinavia",
    "Africa", "South America", "Europe",
]

PARTNER_PHRASES = [
    "a multinational manufacturer",
    "an international consortium",
    "a top-tier research institute",
    "a leading cloud platform",
    "a major logistics provider",
    "a global retailer",
    "a national infrastructure agency",
]

DOLLAR_RE = re.compile(r"\$\d+(?:\.\d+)?\s?[BMK]?")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?\s?%")
COMPANY_RE = re.compile(r"^([A-Z][A-Za-z]*(?:\s+[A-Z][A-Za-z]*){0,2})\b")

SECTOR_LOOKUP = sorted(SECTOR_PHRASES, key=len, reverse=True)
REGION_LOOKUP = sorted(REGIONS, key=len, reverse=True)

def _find_phrase(text: str, phrases: list[str]) -> str:
    low = text.lower()
    for phrase in phrases:
        if phrase.lower() in low:
            return phrase
    return ""

def extract_template(text: str) -> str:
    t = text
    t = re.sub(r"\$\d+(?:\.\d+)?\s?[BMK]?", "$X", t)
    t = re.sub(r"\d+(?:\.\d+)?\s?%", "N%", t)
    t = re.sub(r"\b\d+(?:\.\d+)?\b", "N", t)
    t = re.sub(r"^(?:[A-Z][A-Za-z]+\s){1,3}", "<COMPANY> ", t)
    t = re.sub(r"^(<COMPANY>\s+).+?\s+steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)steps down unexpectedly citing\s+.+$", r"\1<ROLE> steps down unexpectedly citing <REASON>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)(?:.+?\s+)?addresses investor concerns in open letter$", r"\1<ROLE> addresses investor concerns in open letter", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)to present at\s+.+$", r"\1to present at <EVENT>", t, flags=re.IGNORECASE)
    t = re.sub(r"^(<COMPANY>\s+)confirms participation in\s+.+$", r"\1confirms participation in <EVENT>", t, flags=re.IGNORECASE)
    for p in PARTNER_PHRASES:
        t = re.sub(re.escape(p), "<PARTNER>", t, flags=re.IGNORECASE)
    for s in SECTOR_PHRASES:
        t = re.sub(re.escape(s), "<SECTOR>", t, flags=re.IGNORECASE)
    for r in REGIONS:
        t = re.sub(re.escape(r), "<REGION>", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip()
    return t

template_to_index = {}
index_to_template = {}
with TEMPLATE_FILE.open("r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        line = line.rstrip("\n")
        if not line:
            continue
        _, template = line.split("\t", 1)
        template_to_index[template] = idx
        index_to_template[idx] = template

def parse_title(title: str) -> dict:
    title = str(title)
    template = extract_template(title)
    template_index = template_to_index.get(template, -1)

    m_pct = PERCENT_RE.search(title)
    m_dol = DOLLAR_RE.search(title)
    m_company = COMPANY_RE.match(title.strip())

    record = {
        "title": title,
        "template_index": int(template_index),
        "template_text": index_to_template.get(template_index, ""),
        "percentage": m_pct.group(0).strip() if m_pct else "",
        "dollar": m_dol.group(0).strip() if m_dol else "",
        "company": m_company.group(1).strip() if m_company else "",
        "sector": _find_phrase(title, SECTOR_LOOKUP),
        "region": _find_phrase(title, REGION_LOOKUP),
    }
    return record

MAX_TITLES = None  # set to int for testing
OUTPUT_NAME_MAP = {
    "train": "title_template_slot_extractions_parser.json",
    "public": "title_template_slot_extractions_parser_public.json",
    "private": "title_template_slot_extractions_parser_private.json",
}
OUTPUT_PATH = Path(OUTPUT_NAME_MAP[DATASET])

work_df = SOURCE_DF.dropna(subset=["headline"]).copy().reset_index(drop=True)
if MAX_TITLES is not None:
    work_df = work_df.head(MAX_TITLES)

results = []
for i, row in work_df.iterrows():
    rec = parse_title(row["headline"])
    rec["row_index"] = int(i)
    if "session" in row.index and pd.notna(row["session"]):
        rec["session"] = int(row["session"])
    if "bar_ix" in row.index and pd.notna(row["bar_ix"]):
        rec["bar_ix"] = int(row["bar_ix"])
    results.append(rec)

OUTPUT_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"Dataset={DATASET} | Saved {len(results)} records to {OUTPUT_PATH.resolve()}")
pd.DataFrame(results).head()


Saved 17371 records to /Users/lukawedegartner/Documents/eth/Datathon/title_template_slot_extractions_parser_public.json


,title,template_index,template_text,percentage,dollar,company,sector,region,row_index,session,bar_ix
0,Relvos Biosciences opens new office in Southea...,33,<COMPANY> opens new office in <REGION>,,,Relvos Biosciences,,Southeast Asia,0,0,6
1,Orevex Renewables secures $500M contract with ...,0,<COMPANY> secures $X contract with <PARTNER>,,$500M,Orevex Renewables,,,1,0,12
2,Relvos Biosciences names new head of precision...,11,<COMPANY> names new head of <SECTOR> division,,,Relvos Biosciences,precision manufacturing,,2,0,14
3,Calvis Sciences secures $650M contract with a ...,0,<COMPANY> secures $X contract with <PARTNER>,,$650M,Calvis Sciences,,,3,0,20
4,Yorvov Pharmaceuticals secures $180M contract ...,0,<COMPANY> secures $X contract with <PARTNER>,,$180M,Yorvov Pharmaceuticals,,,4,0,22


In [23]:
headline_preprocessed = pd.read_parquet("headline_features.parquet")
len(headline_preprocessed)

17371